# LD and conditional analysis of the convergent case-control hits

For the convergent regions with more than one case-control hit: r² between the hits and a stepwise conditional analysis (conditional P-value).

## Setup, imports and configuration

In [ ]:
import subprocess
import numpy as np, pandas as pd
from pathlib import Path

# Tool
PLINK2 = "/home/jupyter/tools/plink2"

# Paths
DIR_WSPS = "/home/jupyter/workspace/ws_files"
DIR_NOVA = f"{DIR_WSPS}/Novalis_v3_R12"
DIR_RESU = f"{DIR_NOVA}/Results_union"
HAR_LIST_FILE = f"{DIR_WSPS}/HARS_files/HARs_merged/HAR_list_phase_1_union.tsv"

# Ancestry / dataset
ANC, DS = "EUR", "WGS"

# Inputs: outputs of the case-control and convergent variants notebooks
CC_FILE       = f"{DIR_RESU}/CaseControl_{ANC}_{DS}_FDRv3.tsv"
VARIANTS_FILE = f"{DIR_RESU}/Convergent_variants_{ANC}_{DS}.tsv"

# Output folder
TMP = Path(f"{DIR_NOVA}/ConditionalAnalysis"); TMP.mkdir(exist_ok=True)

# Significance thresholds
N_HARS     = sum(1 for _ in open(HAR_LIST_FILE))
BONF       = 0.05 / N_HARS
FDR_THRESH = 0.05

# Model and filters, same as the case-control notebook
MAX_PCS     = 5
BASE_COVARS = ["SEX", "AGE"]
COVAR_NAME  = ",".join(BASE_COVARS + [f"PC{i}" for i in range(1, MAX_PCS + 1)])
MAC_MIN     = 5
GENO_MAX    = 0.05
HWE_P       = 1e-6

# Conditional analysis
MAX_STEPS = 5

## Convergent regions with more than one case-control hit

In [ ]:
cc = pd.read_csv(CC_FILE, sep="\t")
convergent = set(pd.read_csv(VARIANTS_FILE, sep="\t")["HAR"])

# Case-control hits (Bonferroni + FDR-BH) inside the convergent regions
hits = cc[(cc["P"] < BONF) & (cc["FDR_BH"] < FDR_THRESH) & cc["HAR"].isin(convergent)].copy()
n_hits = hits.groupby("HAR").size()
hits = hits[hits["HAR"].isin(n_hits[n_hits > 1].index)]

print(f"{len(convergent)} convergent regions | {len(hits)} hits in "
      f"{hits['HAR'].nunique()} regions with more than one hit")
n_hits[n_hits > 1]

## Covariate file

In [ ]:
# clean covariate file for plink2 (same logic as the case-control notebook)
cov = pd.read_csv(f"{DIR_WSPS}/Working_{DS}_v3/{ANC}/InputFiles/{ANC}_covariate_file.txt",
                  sep=r"\s+", engine="python")
iid = next(c for c in cov.columns if c.upper().lstrip("#") == "IID")
cov[cov.columns[0]] = cov[iid]
keep = list(dict.fromkeys([cov.columns[0], iid, "PHENO"] + COVAR_NAME.split(",")))
cov = cov[keep]
cov["PHENO"] = pd.to_numeric(cov["PHENO"], errors="coerce").fillna(-9).astype(int)
covf = str(TMP / "covar_norm.txt")
cov.to_csv(covf, sep="\t", index=False, na_rep="-9")

## plink2 helpers

In [ ]:
def plink(args):
    return subprocess.run([PLINK2] + args, capture_output=True, text=True)

def ensure_pgen(har):
    pre = f"{DIR_RESU}/CaseControl_{ANC}_{DS}/{har}"
    if not Path(pre + ".pgen").exists():
        vcf = f"{DIR_WSPS}/results_region_extrac_v3_{DS}/{ANC}/InputFiles/Indiv_HARS/{har}.vcf.gz"
        plink(["--vcf", vcf, "--make-pgen", "--double-id", "--max-alleles", "2", "--out", pre])
    return pre

def glm(pre, out, cond_ids=None):
    cmd = ["--pfile", pre, "--glm", "firth-fallback", "hide-covar", "cols=+orbeta,+ci",
           "--pheno", covf, "--pheno-name", "PHENO", "--covar", covf, "--covar-name", COVAR_NAME,
           "--covar-variance-standardize", "--mac", str(MAC_MIN), "--geno", str(GENO_MAX),
           "--hwe", str(HWE_P), "keep-fewhet", "--ci", "0.95", "--out", out]
    if cond_ids:
        lf = out + "_cond.txt"
        Path(lf).write_text("\n".join(cond_ids) + "\n")
        cmd += ["--condition-list", lf]
    r = plink(cmd)
    dfs = []
    for suf in ("logistic.hybrid", "firth.hybrid"):
        f = Path(f"{out}.PHENO.glm.{suf}")
        if f.exists() and f.stat().st_size > 0:
            d = pd.read_csv(f, sep="\t")
            dfs.append(d[d["TEST"] == "ADD"])
    if not dfs:
        raise RuntimeError((r.stdout + r.stderr)[-400:])
    return pd.concat(dfs, ignore_index=True)

## LD (r²) and conditional analysis per region

In [ ]:
summary, hit_steps = [], []
for har, g in hits.groupby("HAR"):
    pre = ensure_pgen(har)
    ids = g.sort_values("P")["ID"].tolist()
    print(f"\n{'='*80}\n{har}  |  {len(ids)} hit(s)")

    # (a) LD among the hits
    idf = TMP / f"{har}_ids.txt"; idf.write_text("\n".join(ids) + "\n")
    plink(["--pfile", pre, "--extract", str(idf), "--export", "A", "--out", str(TMP / f"{har}_geno")])
    G = pd.read_csv(TMP / f"{har}_geno.raw", sep="\t")
    G = G.drop(columns=["FID", "IID", "PAT", "MAT", "SEX", "PHENOTYPE"])
    G.columns = [c.rsplit("_", 1)[0] for c in G.columns]
    r2 = G.corr() ** 2
    r2.to_csv(TMP / f"LD_r2_{har}.tsv", sep="\t")
    print("r2 between hits:")
    print(r2.round(2).to_string())

    # (b) stepwise conditional analysis
    cond = [ids[0]]
    for step in range(1, MAX_STEPS + 1):
        res = glm(pre, str(TMP / f"{har}_s{step}"), cond)
        res = res[~res["ID"].isin(cond)]
        thr = 0.05 / len(res)
        top = res.loc[res["P"].idxmin()]
        others = res[res["ID"].isin(ids)][["ID", "OR", "P"]]
        print(f"\n step {step}: conditioned on {cond} | threshold {thr:.1e}")
        if len(others):
            print(others.to_string(index=False))
        summary.append(dict(HAR=har, step=step, conditioned_on=";".join(cond),
                            best_remaining=top["ID"], P_cond=top["P"], threshold=thr))
        hit_steps += [dict(HAR=har, step=step, conditioned_on=";".join(cond),
                           ID=r["ID"], OR=r["OR"], P_cond=r["P"], r2_with_lead=r2.loc[r["ID"], ids[0]])
                      for _, r in others.iterrows()]
        if top["P"] < thr:
            cond.append(top["ID"])
        else:
            break

## Save results

In [ ]:
out = pd.DataFrame(summary)
out.to_csv(TMP / "conditional_analysis_summary.tsv", sep="\t", index=False)
pd.DataFrame(hit_steps).to_csv(TMP / "conditional_analysis_hits_by_step.tsv", sep="\t", index=False)
print(out.to_string(index=False))